In [ ]:
import csv
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import ipywidgets as widgets
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
def preprocessing():
    # Load the data
    df = pd.read_csv("insurance.csv")
    Charge = np.array(df['charges'])
    Charge_mean = np.mean(Charge)
    Charge_std = np.std(Charge)
    # Define insurance classes with labels "A", "B", "C"
    df['insurance_class'] = pd.cut(df['charges'], bins=[-float('inf'), Charge_mean - (1/2)*Charge_std, Charge_mean + (1/2)*Charge_std, float('inf')], labels=["A", "B", "C"])
    #變成mean+-std之後 訓練的classification range有機會改變 (新增資料影響mean, std)

    print("After adding class information...")
    #畫圖 -- 全體分布圖 (訓練之前)
    bins = np.arange(np.min(Charge),np.max(Charge),1000)
    hist, bin_edges = np.histogram(Charge, bins=bins)
    '''
    def draw_dotted_lines(plt, x):
      plt.plot([x, x], [0, 100], linestyle='dotted', color='red')
    plt.bar(bin_edges[:-1], hist, width=5, color='b', edgecolor='black')
    plt.xlabel('Insurace charges')
    plt.ylabel('Frequency')
    draw_dotted_lines(plt,Charge_mean-(1/2)*(Charge_std))
    draw_dotted_lines(plt,Charge_mean+(1/2)*(Charge_std))
    plt.title('Distribution of Charges')
    print(df.iloc[:10,:])
    '''

    # Convert class labels to integers for model training
    class_mapping = {"A": 0, "B": 1, "C": 2}
    df['insurance_class_encoded'] = df['insurance_class'].map(class_mapping)

    scaler = StandardScaler()
    df[['age', 'bmi', 'children']] = scaler.fit_transform(df[['age', 'bmi', 'children']])
    df['sex'] = df['sex'].apply(lambda x: 1 if x == "male" else 0)
    df['smoker'] = df['smoker'].apply(lambda x: 1 if x == "yes" else 0)
    region_mapping = {"southwest": 0.25, "southeast": 0.5, "northwest": 0.75, "northeast": 1.0}
    df['region'] = df['region'].map(region_mapping)

    # Split data into training and testing sets
    x = df[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
    y = df['insurance_class_encoded']
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    # Convert data to tensors
    x_train = torch.tensor(x_train.values, dtype=torch.float32)
    x_test = torch.tensor(x_test.values, dtype=torch.float32)
    y_train = torch.tensor(y_train.values, dtype=torch.long)
    y_test = torch.tensor(y_test.values, dtype=torch.long)

    return x_train, x_test, y_train, y_test


In [ ]:
# Define Wide & Deep Model
class WideAndDeep(nn.Module):
    def __init__(self, input_dim, deep_hidden_dim, num_classes):
        super(WideAndDeep, self).__init__()
        # Wide part
        self.wide = nn.Linear(input_dim, num_classes)

        # Deep part
        self.deep = nn.Sequential(
            nn.Linear(input_dim, deep_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(deep_hidden_dim, deep_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(deep_hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        wide_out = self.wide(x)
        deep_out = self.deep(x)
        return wide_out + deep_out

In [ ]:
x_train, x_test, y_train, y_test = preprocessing()
# Initialize model, optimizer, and loss function
input_dim = x_train.shape[1]
deep_hidden_dim = 64
num_classes = 3

model = WideAndDeep(input_dim=input_dim, deep_hidden_dim=deep_hidden_dim, num_classes=num_classes)


After adding class information...


In [ ]:
# Training loop
def train_model(model, x_train, y_train, x_test, y_test, epochs):
    #Loss = [] #Record Train Loss
    #Acc = [] #Record Train Acc
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        y_logits = model(x_train)
        y_preds = torch.argmax(y_logits, dim=-1)

        # Compute loss and accuracy
        loss = loss_fn(y_logits, y_train)
        acc = (y_preds == y_train).sum().item() / len(y_train)
        #Loss.append(loss.detach().numpy())
        #Acc.append(acc)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Evaluation on test set
        model.eval()
        with torch.no_grad():
            test_logits = model(x_test)
            test_preds = torch.argmax(test_logits, dim=-1)
            test_loss = loss_fn(test_logits, y_test)
            test_acc = (test_preds == y_test).sum().item() / len(y_test)

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}: Train Loss: {loss:.4f}, Train Acc: {acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
# Train the model
train_model(model, x_train, y_train, x_test, y_test, epochs=2000)

Epoch 0: Train Loss: 1.4741, Train Acc: 0.2065, Test Loss: 1.4391, Test Acc: 0.2015
Epoch 10: Train Loss: 1.3508, Train Acc: 0.2140, Test Loss: 1.3186, Test Acc: 0.2276
Epoch 20: Train Loss: 1.2341, Train Acc: 0.2561, Test Loss: 1.2050, Test Acc: 0.2761
Epoch 30: Train Loss: 1.1213, Train Acc: 0.3308, Test Loss: 1.0800, Test Acc: 0.3843
Epoch 40: Train Loss: 0.9960, Train Acc: 0.4916, Test Loss: 0.9404, Test Acc: 0.5821
Epoch 50: Train Loss: 0.8699, Train Acc: 0.6383, Test Loss: 0.8049, Test Acc: 0.7687
Epoch 60: Train Loss: 0.7707, Train Acc: 0.6916, Test Loss: 0.7041, Test Acc: 0.7388
Epoch 70: Train Loss: 0.7085, Train Acc: 0.7290, Test Loss: 0.6351, Test Acc: 0.7836
Epoch 80: Train Loss: 0.6463, Train Acc: 0.7617, Test Loss: 0.5755, Test Acc: 0.8172
Epoch 90: Train Loss: 0.6046, Train Acc: 0.8000, Test Loss: 0.5226, Test Acc: 0.8396
Epoch 100: Train Loss: 0.5534, Train Acc: 0.8290, Test Loss: 0.4792, Test Acc: 0.8769
Epoch 110: Train Loss: 0.5163, Train Acc: 0.8411, Test Loss: 0.44

In [ ]:
# 通用數據標準化函數
def standardize_data(data, means, stds):
    """標準化數據."""
    standardized = (data - means) / stds
    return standardized

# 預測函數
def predict_and_display(data, model, prediction_labels):
    """進行預測並顯示結果."""
    model.eval()
    with torch.no_grad():
        prediction = model(data)
        predicted_class = torch.argmax(prediction, dim=1).item()
    print(f"Prediction: {prediction_labels[predicted_class]}")
    return predicted_class

count = [9] # 用於計數，以便在每10次反饋後重新訓練模型
# 通用反饋函數
def feedback_mechanism(predicted_class, prediction_labels):
    """反饋機制."""
    Reflection = widgets.Dropdown(
        options=[('Yes, correct', 1), ('No, incorrect', -1)],
        description='Is it correct?'
    )
    display(Reflection)

    def save_feedback(b):
        feedback_value = Reflection.value
        if feedback_value == 1:
            print("Thank you for your feedback. Have a great day!")
        elif feedback_value == -1:
            print("Thank you for your feedback. Please enter the correct amount:")

            options = widgets.FloatText()
            display(options)


            def save_correct_amount(c):
                correct_amount = options

                # 保存正確答案及輸入的數據到文件
                user_data = [
                    age.value, sex.label, bmi.value, children.value, smoker.label, region.label,
                    options.value,
                ]
                with open("insurance.csv", mode='a', newline='') as file:
                    writer = csv.writer(file)
                    writer.writerow(user_data)
                print(f"Feedback saved. Correct: {correct_amount.value}. Data saved to 'insurance.csv'.")
                print("We will update our model accordingly. Thank you!")

                count[0] += 1
                #print("count: ", count[0])
                if count[0] == 10:
                    x_train, x_test, y_train, y_test = preprocessing()
                    model = WideAndDeep(input_dim=input_dim, deep_hidden_dim=deep_hidden_dim, num_classes=num_classes)
                    train_model(model, x_train, y_train, x_test, y_test, epochs=2000)
                    count[0] = 0

            amount_save_button = widgets.Button(description="Save Correct Amount")
            amount_save_button.on_click(save_correct_amount)
            display(amount_save_button)

    save_button = widgets.Button(description="Save Feedback")
    save_button.on_click(save_feedback)
    display(save_button)

# 標準化參數
df = pd.read_csv("insurance.csv")
Charge = np.array(df['charges'])
Charge_mean = np.mean(Charge)
Charge_std = np.std(Charge)
means = np.array([40.0, 30.0, 1.0])  # Example mean values for age, bmi, children
stds = np.array([14.0, 6.0, 1.0])    # Example std values for age, bmi, children
prediction_labels = ["A: Below $%.2f"%(Charge_mean-(1/2)*(Charge_std)), "B: $%.2f-$%.2f"%((Charge_mean-(1/2)*(Charge_std)),(Charge_mean+(1/2)*(Charge_std))), "C: Above $%.2f"%(Charge_mean+(1/2)*(Charge_std))]

df = pd.read_csv("insurance.csv")

# 使用者輸入
age = widgets.IntText(description='Age:')
sex = widgets.Dropdown(options=[('male', 1), ('female', 0)], description='Sex:')
bmi = widgets.FloatText(description='BMI:')
children = widgets.IntText(description='Children:')
smoker = widgets.Dropdown(options=[('yes', 1), ('no', 0)], description='Smoker:')
region = widgets.Dropdown(options=[('southwest', 0.25), ('southeast', 0.5), ('northwest', 0.75), ('northeast', 1.0)], description='Region:')
mode = widgets.Dropdown(options=[('User', 1), ('Supervisor', 0)], description='Mode:')
display(age, sex, bmi, children, smoker, region, mode)

predict_button = widgets.Button(description="Predict")

# 預測按鈕邏輯
def make_prediction(b):
    # 收集用戶數據
    user_input = np.array([
        age.value,
        bmi.value,
        children.value
    ])
    # 標準化數據
    standardized_input = standardize_data(user_input, means, stds)

    # 組裝完整數據 (加上非標準化特徵)
    full_input = torch.tensor([
        [
            standardized_input[0],  # Age
            sex.value,
            standardized_input[1],  # BMI
            standardized_input[2],  # Children
            smoker.value,
            region.value,
        ]
    ], dtype=torch.float32)

    # 預測並顯示
    predicted_class = predict_and_display(full_input, model, prediction_labels)

    # 啟動反饋機制
    if mode.value == 0:
        feedback_mechanism(predicted_class, prediction_labels)

predict_button.on_click(make_prediction)
display(predict_button)


IntText(value=0, description='Age:')

Dropdown(description='Sex:', options=(('male', 1), ('female', 0)), value=1)

FloatText(value=0.0, description='BMI:')

IntText(value=0, description='Children:')

Dropdown(description='Smoker:', options=(('yes', 1), ('no', 0)), value=1)

...

Button(description='Save Feedback', style=ButtonStyle())

Thank you for your feedback. Please enter the correct amount:


FloatText(value=0.0)

Button(description='Save Correct Amount', style=ButtonStyle())

Feedback saved. Correct: 8500.0. Data saved to 'insurance.csv'.
We will update our model accordingly. Thank you!
After adding class information...
Epoch 0: Train Loss: 1.5008, Train Acc: 0.2726, Test Loss: 1.4760, Test Acc: 0.2687
Epoch 10: Train Loss: 1.3412, Train Acc: 0.2773, Test Loss: 1.3189, Test Acc: 0.2836
Epoch 20: Train Loss: 1.1902, Train Acc: 0.3016, Test Loss: 1.1695, Test Acc: 0.3022
Epoch 30: Train Loss: 1.0594, Train Acc: 0.4678, Test Loss: 1.0315, Test Acc: 0.5149
Epoch 40: Train Loss: 0.9492, Train Acc: 0.5892, Test Loss: 0.9236, Test Acc: 0.6455
Epoch 50: Train Loss: 0.8580, Train Acc: 0.6452, Test Loss: 0.8370, Test Acc: 0.6754
Epoch 60: Train Loss: 0.7730, Train Acc: 0.7003, Test Loss: 0.7690, Test Acc: 0.7201
Epoch 70: Train Loss: 0.7227, Train Acc: 0.7330, Test Loss: 0.7187, Test Acc: 0.7313
Epoch 80: Train Loss: 0.6921, Train Acc: 0.7292, Test Loss: 0.6793, Test Acc: 0.7537
Epoch 90: Train Loss: 0.6434, Train Acc: 0.7740, Test Loss: 0.6410, Test Acc: 0.7687
Epoc